# Catalog and table setup
Creates the Unity Catalog schemas (`bronze`, `silver`, `gold`, `ops`) and initializes empty Delta tables for the pipeline.

In [0]:
from pyspark.sql import functions as F

# Session and catalog configuration
spark.conf.set("spark.sql.session.timeZone", "UTC")
CATALOG = spark.sql("SELECT current_catalog()").first()[0]

# Create schemas
for schema_name in ["bronze", "silver", "gold", "ops"]:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{schema_name}")

# Bronze tables (raw API payloads and metadata)
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.bronze.electricity_price_api_raw (
    bronze_record_id STRING NOT NULL,
    source_system STRING NOT NULL,
    request_key STRING NOT NULL,
    request_date DATE NOT NULL,
    price_area STRING,
    station_id STRING,
    source_url STRING NOT NULL,
    http_status INT NOT NULL,
    payload_json STRING,
    payload_hash STRING NOT NULL,
    record_count INT NOT NULL,
    error_message STRING,
    retrieved_at_utc TIMESTAMP NOT NULL,
    ingestion_run_id STRING NOT NULL
) USING DELTA
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.bronze.weather_api_raw
LIKE {CATALOG}.bronze.electricity_price_api_raw
""")

# Silver tables (parsed and joined hourly data)
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.silver.electricity_price_hourly (
    price_area STRING NOT NULL,
    time_start_utc TIMESTAMP NOT NULL,
    time_end_utc TIMESTAMP,
    time_start_local TIMESTAMP,
    nok_per_kwh DOUBLE,
    eur_per_kwh DOUBLE,
    exchange_rate DOUBLE,
    source_bronze_record_id STRING,
    source_retrieved_at_utc TIMESTAMP,
    processed_at_utc TIMESTAMP
) USING DELTA
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.silver.weather_hourly (
    station_id STRING NOT NULL,
    time_start_utc TIMESTAMP NOT NULL,
    time_start_local TIMESTAMP,
    air_temperature_c DOUBLE,
    wind_speed_mps DOUBLE,
    precipitation_1h_mm DOUBLE,
    temperature_quality_code INT,
    wind_quality_code INT,
    precipitation_quality_code INT,
    source_bronze_record_id STRING,
    source_retrieved_at_utc TIMESTAMP,
    processed_at_utc TIMESTAMP
) USING DELTA
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.silver.weather_price_hourly (
    price_area STRING NOT NULL,
    station_id STRING NOT NULL,
    time_start_utc TIMESTAMP NOT NULL,
    time_end_utc TIMESTAMP,
    time_start_local TIMESTAMP,
    nok_per_kwh DOUBLE,
    eur_per_kwh DOUBLE,
    exchange_rate DOUBLE,
    air_temperature_c DOUBLE,
    wind_speed_mps DOUBLE,
    precipitation_1h_mm DOUBLE,
    weather_match_status STRING,
    processed_at_utc TIMESTAMP
) USING DELTA
""")

# Gold and Ops tables
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.gold.price_by_temperature_band (
    price_area STRING NOT NULL,
    temperature_band_order INT NOT NULL,
    temperature_band STRING NOT NULL,
    observation_count BIGINT,
    average_price_nok_per_kwh DOUBLE,
    median_price_nok_per_kwh DOUBLE,
    minimum_price_nok_per_kwh DOUBLE,
    maximum_price_nok_per_kwh DOUBLE,
    average_wind_speed_mps DOUBLE,
    updated_at_utc TIMESTAMP
) USING DELTA
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.ops.data_quality_results (
    run_id STRING NOT NULL,
    layer STRING NOT NULL,
    check_name STRING NOT NULL,
    severity STRING NOT NULL,
    passed BOOLEAN NOT NULL,
    actual_value DOUBLE,
    message STRING,
    checked_at_utc TIMESTAMP NOT NULL
) USING DELTA
""")

for schema_name in ["bronze", "silver", "gold", "ops"]:
    print(f"Created or verified schema: {CATALOG}.{schema_name}")